In [1]:
# ============================================================
# 1. IMPORTS AND SETUP
# ============================================================

import sys
from pathlib import Path
import pandas as pd
import importlib

project_root = Path(r"C:\Users\HUGO\Desktop\Q8 - NORUEGA\TFG\tfg")

sys.path.append(str(project_root))

from src.data_pipeline import DatabaseManager, DataLoader, DataPreprocessor

import src.events
importlib.reload(src.events)

from src.events import SimpleEventDetector, CombinedEventDetector

db_path = project_root / "NordPoool" / "data" / "thesis_database.db"

db = DatabaseManager(db_path)
loader = DataLoader(db)
pre = DataPreprocessor()

simple_detector = SimpleEventDetector()
combined_detector = CombinedEventDetector()

print("Setup completed")

Setup completed


In [2]:
# ============================================================
# 2. LOAD DATA
# ============================================================

df_prices = loader.load_prices()
df_volumes = loader.load_volumes()
df_flows = loader.load_flows()
df_capacities = loader.load_capacities()
df_zones = loader.load_bidding_zones()

print("Prices:", df_prices.shape)
print("Volumes:", df_volumes.shape)
print("Flows:", df_flows.shape)
print("Capacities:", df_capacities.shape)
print("Zones:", df_zones.shape)

Prices: (1007980, 5)
Volumes: (350880, 6)
Flows: (230470, 6)
Capacities: (35132, 7)
Zones: (20, 4)


In [3]:
# ============================================================
# 3. CREATE DATETIME INDEX
# ============================================================

prices_dt = pre.create_datetime_index(df_prices, convert_to_utc=False)
volumes_dt = pre.create_datetime_index(df_volumes, convert_to_utc=False)
flows_dt = pre.create_datetime_index(df_flows, convert_to_utc=False)
capacities_dt = pre.create_datetime_index(df_capacities, convert_to_utc=False)

print("Prices:", prices_dt.index.min(), "->", prices_dt.index.max())
print("Volumes:", volumes_dt.index.min(), "->", volumes_dt.index.max())
print("Flows:", flows_dt.index.min(), "->", flows_dt.index.max())
print("Capacities:", capacities_dt.index.min(), "->", capacities_dt.index.max())

Prices: 2020-01-01 00:00:00 -> 2025-09-30 23:00:00
Volumes: 2020-01-01 00:00:00 -> 2021-12-31 23:00:00
Flows: 2020-01-01 00:00:00 -> 2020-12-31 23:00:00
Capacities: 2020-01-01 00:00:00 -> 2020-12-31 23:00:00


In [4]:
# ============================================================
# 4. MERGE PRICES AND VOLUMES
# ============================================================

prices_base = prices_dt.reset_index()
volumes_base = volumes_dt.reset_index()

node_df = prices_base.merge(
    volumes_base[
        [
            "datetime",
            "zone_id",
            "buy_volume_value",
            "sell_volume_value"
        ]
    ],
    on=["datetime", "zone_id"],
    how="inner"
)

print("Node dataset shape:", node_df.shape)

display(node_df.head())

Node dataset shape: (350960, 8)


,datetime,price_id,zone_id,delivery_day,hour,price_value,buy_volume_value,sell_volume_value
0,2020-01-01,1,1,2020-01-01,0,28.78,688.8,283.8
1,2020-01-01,6,6,2020-01-01,0,41.88,0.3,462.9
2,2020-01-01,8,8,2020-01-01,0,41.88,814.3,243.6
3,2020-01-01,7,7,2020-01-01,0,41.88,0.0,885.0
4,2020-01-01,2,2,2020-01-01,0,28.78,1005.3,1703.4


In [5]:
# ============================================================
# 5. DETECT SIMPLE NODE EVENTS
# ============================================================

node_events = simple_detector.detect_price_events(node_df)
node_events = simple_detector.detect_volume_events(node_events)

display(node_events.head())

,datetime,price_id,zone_id,delivery_day,hour,price_value,buy_volume_value,sell_volume_value,price_delta,abs_price_delta,...,low_generation,volume_imbalance,strong_buy_pressure,strong_sell_pressure,buy_volume_delta,sell_volume_delta,abs_buy_volume_delta,abs_sell_volume_delta,buy_volume_spike,sell_volume_spike
0,2020-01-01 00:00:00,1,1,2020-01-01,0,28.78,688.8,283.8,NaN,NaN,...,False,0.416410,False,False,NaN,NaN,NaN,NaN,False,False
25,2020-01-01 01:00:00,21,1,2020-01-01,1,28.45,680.7,284.8,-0.33,0.33,...,False,0.410047,False,False,-8.1,1.0,8.1,1.0,False,False
41,2020-01-01 02:00:00,41,1,2020-01-01,2,27.90,669.8,281.2,-0.55,0.55,...,False,0.408623,False,False,-10.9,-3.6,10.9,3.6,False,False
63,2020-01-01 03:00:00,61,1,2020-01-01,3,27.52,666.4,266.2,-0.38,0.38,...,False,0.429123,False,False,-3.4,-15.0,3.4,15.0,False,False
89,2020-01-01 04:00:00,81,1,2020-01-01,4,27.54,663.0,252.6,0.02,0.02,...,False,0.448231,False,False,-3.4,-13.6,3.4,13.6,False,False


In [7]:
# ============================================================
# 6. DETECT COMBINED NODE EVENTS
# ============================================================

combined_node_events = combined_detector.detect_combined_events(node_events)

combined_node_event_columns = [
    "generation_surplus",
    "demand_pressure",
    "strong_demand_pressure",
    "strong_generation_pressure",
    "demand_driven_price_spike",
    "generation_driven_low_price",
    "scarcity_price_event",
    "oversupply_price_event",
    "buy_pressure_price_spike",
    "sell_pressure_low_price",
    "price_separation",
    "extreme_price_separation",
    "zone_price_outlier",
]

combined_node_event_counts = (
    combined_node_events[combined_node_event_columns]
    .sum()
    .reset_index()
)

combined_node_event_counts.columns = ["event_type", "count"]

display(combined_node_event_counts)

,event_type,count
0,generation_surplus,2117
1,demand_pressure,1070
2,strong_demand_pressure,4878
3,strong_generation_pressure,14048
4,demand_driven_price_spike,4915
5,generation_driven_low_price,2589
6,scarcity_price_event,139
7,oversupply_price_event,600
8,buy_pressure_price_spike,1372
9,sell_pressure_low_price,5266


In [8]:
# ============================================================
# 7. DETECT SIMPLE EDGE EVENTS
# ============================================================

flow_events = simple_detector.detect_flow_events(
    flows_dt.reset_index()
)

capacity_events = simple_detector.detect_capacity_events(
    capacities_dt.reset_index()
)

print("Flow events:", flow_events.shape)
print("Capacity events:", capacity_events.shape)

display(flow_events.head())
display(capacity_events.head())

Flow events: (230470, 15)
Capacity events: (35132, 16)


,datetime,flow_id,from_zone_id,to_zone_id,delivery_day,hour,flow_value,abs_flow_value,low_flow,high_flow,flow_delta,abs_flow_delta,flow_spike,flow_reversal,has_flow_event
0,2020-11-18 00:00:00,200903,7,13,2020-11-18,0,0.0,0.0,False,False,NaN,NaN,False,False,False
1,2020-11-18 01:00:00,200931,7,13,2020-11-18,1,0.0,0.0,False,False,0.0,0.0,False,False,False
2,2020-11-18 02:00:00,200959,7,13,2020-11-18,2,0.0,0.0,False,False,0.0,0.0,False,False,False
3,2020-11-18 03:00:00,200987,7,13,2020-11-18,3,0.0,0.0,False,False,0.0,0.0,False,False,False
4,2020-11-18 04:00:00,201015,7,13,2020-11-18,4,0.0,0.0,False,False,0.0,0.0,False,False,False


,datetime,capacity_id,capacity_code,from_zone_id,to_zone_id,delivery_day,hour,capacity_value,abs_capacity_value,low_capacity,high_capacity,capacity_delta,abs_capacity_delta,capacity_spike,capacity_drop,has_capacity_event
0,2020-01-01 00:00:00,1,NO1->NO3,12,14,2020-01-01,0,200.0,200.0,False,False,NaN,NaN,False,False,False
4,2020-01-01 01:00:00,5,NO1->NO3,12,14,2020-01-01,1,200.0,200.0,False,False,0.0,0.0,False,False,False
9,2020-01-01 02:00:00,9,NO1->NO3,12,14,2020-01-01,2,200.0,200.0,False,False,0.0,0.0,False,False,False
12,2020-01-01 03:00:00,13,NO1->NO3,12,14,2020-01-01,3,200.0,200.0,False,False,0.0,0.0,False,False,False
17,2020-01-01 04:00:00,17,NO1->NO3,12,14,2020-01-01,4,200.0,200.0,False,False,0.0,0.0,False,False,False


In [9]:
# ============================================================
# 8. MERGE FLOW AND CAPACITY EVENTS
# ============================================================

capacity_cols = [
    "datetime",
    "from_zone_id",
    "to_zone_id",
    "capacity_value",
    "abs_capacity_value",
    "low_capacity",
    "high_capacity",
    "capacity_spike",
    "capacity_drop",
    "has_capacity_event"
]

edge_df = flow_events.merge(
    capacity_events[capacity_cols],
    on=[
        "datetime",
        "from_zone_id",
        "to_zone_id"
    ],
    how="inner"
)

print("Edge dataset shape:", edge_df.shape)

display(edge_df.head())

Edge dataset shape: (35132, 22)


,datetime,flow_id,from_zone_id,to_zone_id,delivery_day,hour,flow_value,abs_flow_value,low_flow,high_flow,...,flow_spike,flow_reversal,has_flow_event,capacity_value,abs_capacity_value,low_capacity,high_capacity,capacity_spike,capacity_drop,has_capacity_event
0,2020-01-01 00:00:00,5,12,14,2020-01-01,0,200.0,200.0,False,True,...,False,False,True,200.0,200.0,False,False,False,False,False
1,2020-01-01 01:00:00,31,12,14,2020-01-01,1,200.0,200.0,False,True,...,False,False,True,200.0,200.0,False,False,False,False,False
2,2020-01-01 02:00:00,57,12,14,2020-01-01,2,200.0,200.0,False,True,...,False,False,True,200.0,200.0,False,False,False,False,False
3,2020-01-01 03:00:00,83,12,14,2020-01-01,3,200.0,200.0,False,True,...,False,False,True,200.0,200.0,False,False,False,False,False
4,2020-01-01 04:00:00,109,12,14,2020-01-01,4,200.0,200.0,False,True,...,False,False,True,200.0,200.0,False,False,False,False,False


In [10]:
# ============================================================
# 9. DETECT EDGE COMBINED EVENTS
# ============================================================

combined_edge_events = combined_detector.detect_edge_combined_events(edge_df)

combined_edge_event_columns = [
    "congestion",
    "high_flow_low_capacity",
    "capacity_drop_with_high_flow",
    "constrained_interconnection",
]

combined_edge_event_counts = (
    combined_edge_events[combined_edge_event_columns]
    .sum()
    .reset_index()
)

combined_edge_event_counts.columns = ["event_type", "count"]

display(combined_edge_event_counts)

,event_type,count
0,congestion,12997
1,high_flow_low_capacity,0
2,capacity_drop_with_high_flow,152
3,constrained_interconnection,13012


In [11]:
# ============================================================
# CHECK CAPACITY USAGE DISTRIBUTION
# ============================================================

display(
    combined_edge_events["capacity_usage"].describe()
)

print("Capacity usage > 1:", (combined_edge_events["capacity_usage"] > 1).sum())
print("Capacity usage >= 0.90:", (combined_edge_events["capacity_usage"] >= 0.90).sum())

count    35132.000000
mean         0.396469
std          0.477367
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000
Name: capacity_usage, dtype: float64

Capacity usage > 1: 0
Capacity usage >= 0.90: 12997


In [12]:
# ============================================================
# CHECK HIGH FLOW AND LOW CAPACITY OVERLAP
# ============================================================

print("high_flow:", combined_edge_events["high_flow"].sum())
print("low_capacity:", combined_edge_events["low_capacity"].sum())
print(
    "high_flow & low_capacity:",
    (combined_edge_events["high_flow"] & combined_edge_events["low_capacity"]).sum()
)

high_flow: 2617
low_capacity: 1262
high_flow & low_capacity: 0
